# BB 지수 (BB Rating) — Run It Yourself

This notebook does everything we walked through in chat, in one place:
load the frozen model, score the full historical file, build the
trailing-36-month reference, and let you score **any race** by
`race_id` or by date + region.

**Before running:** make sure these 3 files exist:
- `code/horse_rating_10var_v2.py`
- `code/fit_engine.py`
- `data/gate2_features_built.csv`
- `params/rating_model_params_10var.json`

If your folder is not `~/Desktop/bb_rating_project`, edit the paths in
the next cell.


## 0. Setup — paths and imports

In [ ]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# ---- EDIT THIS if your project folder is somewhere else ----
PROJECT_DIR = Path.home() / "Desktop" / "bb_rating_project"

CODE_DIR   = PROJECT_DIR / "code"
DATA_DIR   = PROJECT_DIR / "data"
PARAMS_DIR = PROJECT_DIR / "params"

FEATURES_CSV = DATA_DIR / "gate2_features_built.csv"
PARAMS_JSON  = PARAMS_DIR / "rating_model_params_10var.json"

sys.path.insert(0, str(CODE_DIR))
import horse_rating_10var_v2 as hr

print("Project dir:", PROJECT_DIR)
print("Features file exists:", FEATURES_CSV.exists())
print("Params file exists:  ", PARAMS_JSON.exists())

## 1. Load the frozen model

This is the "frozen" part we talked about — `beta_raw`, `mu`, `sigma`.
These do NOT change per race. They only change if someone re-runs
`fit_production()` on new data and overwrites this JSON file.

In [ ]:
params = hr.load_params(str(PARAMS_JSON))
beta_raw = np.array(params["beta_raw"])
mu       = np.array(params["mu"])
sigma    = np.array(params["sigma"])
var_list = params["var_list"]

print("Model fit date:", params.get("fit_date"))
print("Variables (in order):", var_list)

## 2. Load the full historical feature file

This is the raw data pulled from RDS (via `gate2_features_built.csv`).
This step can take a few seconds — it's ~369k rows.

In [ ]:
df = pd.read_csv(FEATURES_CSV)
df["race_date"] = pd.to_datetime(df["race_date"])

print(f"{len(df):,} rows, {df['race_id'].nunique():,} races")
print(f"Date range: {df['race_date'].min().date()} to {df['race_date'].max().date()}")
print(f"Regions: {df['region'].unique()}")

## 3. Score the full history (compute V + block contributions)

This runs the SAME math for every historical horse-row:
`V = ((X - mu) / sigma) @ beta_raw`, then splits it into the 5 blocks
(스피드/전적/기수/경주조건/경험). This is what builds the population
we compare any new horse against.

In [ ]:
scored = hr.compute_V(df, var_list, beta_raw, mu, sigma)
scored = hr.block_contributions(scored, var_list, hr.BLOCKS_10, beta_raw, mu, sigma)

print(f"Scored {len(scored):,} horse-rows (rows with complete 10-var data)")
scored.to_pickle(PROJECT_DIR / "scored_history_cache.pkl")
print("Cached to scored_history_cache.pkl (so you don't have to redo this every time)")

## 4. Build the reference snapshot

This is the trailing-36-month lookup table used to turn a horse's raw
V (or block contribution) into a 0-100 percentile score. It's built
"as of" the latest date in your data -- if your file only goes up to
last week, this snapshot reflects up-to-last-week, not today.

In [ ]:
snapshot = hr.build_reference_snapshot(
    scored, hr.BLOCKS_10, months=hr.ROLLING_MONTHS, min_rows=hr.MIN_REFERENCE_ROWS
)

print("Reference as_of_date:      ", snapshot["as_of_date"])
print("Window (months):           ", snapshot["window_months"])
print("Used expanding fallback?:  ", snapshot["used_expanding_fallback"])
print("Reference population size: ", snapshot["n_reference_rows"])

## 5. Find a race to score

Two ways to find a race:
- **By date + region** (e.g. "last Friday, 서울") -- run the next cell
  to list race_ids on that date.
- **By race_id directly** if you already know it.

Edit `TARGET_DATE` and `TARGET_REGION` below, then run.

In [ ]:
TARGET_DATE = "2026-07-24"     # <-- change this to the date you want (YYYY-MM-DD)
TARGET_REGION = "서울"          # <-- "서울" or "부산"

day_races = df[(df["race_date"] == TARGET_DATE) & (df["region"] == TARGET_REGION)]
print(f"Races found on {TARGET_DATE} in {TARGET_REGION}:")
print(day_races.groupby("race_id").size().rename("n_horses"))

if len(day_races) == 0:
    print()
    print("No races found on that date in this file.")
    print("Your file's latest date is:", df["race_date"].max().date())
    print("If you need a more recent date, you need a fresh RDS pull first --")
    print("this notebook only scores what's already in gate2_features_built.csv.")

## 6. Score the race

Set `TARGET_RACE_ID` to one of the race_ids printed above, then run.

In [ ]:
TARGET_RACE_ID = 202607190104   # <-- change this to the race_id you want to score

race = df[df["race_id"] == TARGET_RACE_ID].copy()
if len(race) == 0:
    raise ValueError(f"race_id {TARGET_RACE_ID} not found in the file")

horses = []
for _, row in race.sort_values("back_num").iterrows():
    h = {"horse_num": int(row["back_num"]), "horse_id": int(row["horse_id"])}
    for v in var_list:
        h[v] = float(row[v])
    horses.append(h)

print(f"Built {len(horses)} horse dicts for race_id {TARGET_RACE_ID}")
print(f"  {race['region'].iloc[0]}, {race['race_date'].iloc[0].date()}, "
      f"{race['race_class'].iloc[0]}, {race['distance'].iloc[0]}m")

results = hr.score_race(
    horses, var_list, hr.BLOCKS_10, beta_raw, mu, sigma,
    reference_snapshot=snapshot
)

print()
print("="*70)
for r in sorted(results, key=lambda x: -x.ability_score):
    print(f"마번 {r.horse_num} (horse_id={r.horse_id})")
    print(f"  종합 능력 점수: {r.ability_score}   우승 기대도(fund_p): {r.win_expectancy:.1%}")
    print(f"  부문별: " + ", ".join(f"{k} {v}점" for k, v in r.sub_scores.items()))
    print(f"  스토리: {r.story}")
    print()

## 7. (Optional) Compare to what actually happened

Only useful for past races where the outcome is already known --
obviously not available for a race that hasn't run yet.

In [ ]:
actual = race.set_index("back_num")[["rank", "win"]].sort_index()
print("ACTUAL RESULT:")
print(actual)

## Notes for handoff (조 / 최)

- **Step 1 (frozen model)** only needs to be redone when someone
  decides to refit -- this does NOT happen automatically. There is no
  cadence decided yet (weekly? monthly?) -- that's an open question to
  raise.
- **Step 2 (feature file)** currently must be manually pulled from RDS
  and manually engineered into the 10 model variables. There is no
  live connection from this notebook to RDS -- it only reads whatever
  CSV sits in `data/`.
- **Step 4 (reference snapshot)** goes stale the moment real time
  passes the `as_of_date` printed above. Rebuilding it just means
  re-running steps 2-4 on a fresher `gate2_features_built.csv`.
- To score TODAY's race card, someone needs to: pull today's runners
  from RDS -> compute their 10 features -> build horse dicts the same
  way Step 6 does -> call `score_race()`. That feature-engineering
  step (raw KRA fields -> AVESPRAT/LSPEDRAT/etc.) is NOT in this
  notebook -- it lives in `build_bc_features.py` / `build_tier1_features.py`.
